# 02 — Build retrieval eval set

Phase C: bootstrap `datasets/retrieval_eval.jsonl` — the ≥200-pair eval set the harness scores against. Plan §B step 1–5:

1. For each source, take items and build a 2–3-sentence summary (LLM, one-shot).
2. Generate candidate queries from each summary (avoiding raw-body phrasing).
3. Reject queries with >3-gram overlap with the source body — aim <5% retained verbatim phrasing.
4. Hand-curate ~30% per source — synonyms, abbreviations, real user-style questions.
5. Add 5–10 cross-source negative pairs per source.
6. Pin the file in git.

Generation is LLM-driven; ragas's `TestsetGenerator` is one option but the dependency lookup is your call. Either way, the output schema is a JSONL of `{query, source, expected_content_id}`.

In [ ]:
import json
import os
from pathlib import Path

from domains.research.sources import ResearchSource
from domains.sessions.sources import SessionsSource
from domains.wiki.sources import LocalFileSource, RawStoreSource

BACKUP = Path(os.environ.get("BACKUP_SOURCE_DIR", "~")).expanduser()
OUT = Path("datasets/retrieval_eval.jsonl")

In [ ]:
# Load items per source.
items_by_source = {
    "raw_store": RawStoreSource(BACKUP / "raw_store.db").get_items(),
    "sessions": SessionsSource(BACKUP / "sessions.db").get_items(),
    "research": ResearchSource(BACKUP / "research.db").get_items(),
    "notes": LocalFileSource(BACKUP / "notes").get_items(),
}
{k: len(v) for k, v in items_by_source.items()}

## Step 1–2: summarize and generate queries

Wire your preferred LLM here. Suggested shape:

```python
candidates = []  # list[(query, source, expected_content_id)]
for source, items in items_by_source.items():
    for item in items[:N]:
        summary = llm_summarize(item.text)             # 2-3 sentences
        for query in llm_generate_queries(summary):    # ~3 per item
            candidates.append((query, source, item.item_id))
```

In [ ]:
# Step 3: n-gram overlap reject.
def ngram_overlap_ratio(query: str, body: str, n: int = 3) -> float:
    def grams(s):
        toks = s.lower().split()
        return {tuple(toks[i : i + n]) for i in range(len(toks) - n + 1)}

    q, b = grams(query), grams(body)
    return len(q & b) / max(len(q), 1)


# Reject if overlap > 0.3 (tune to taste, plan suggests <5% retained verbatim).
# survivors = [c for c in candidates if ngram_overlap_ratio(c[0], item_text(c[2])) <= 0.3]

## Step 4–5: hand-curate + add negatives

Open the survivors in a quick UI (e.g. `pd.DataFrame(survivors).to_csv(...)` and edit) — replace the most LLM-flavored queries with plausible user-style questions, add 5–10 negatives per source.

## Step 6: write JSONL

In [ ]:
# pairs = [...]  # final list[(query, source, expected_content_id)]
# OUT.parent.mkdir(parents=True, exist_ok=True)
# with OUT.open("w") as f:
#     for q, s, eid in pairs:
#         f.write(json.dumps({"query": q, "source": s, "expected_content_id": eid}) + "\n")
# print(f"Wrote {len(pairs)} pairs to {OUT}")